In [1]:
import os
import numpy as np
import pandas as pd

DATA = "../data" if os.path.exists("../data") else "data"

def load(name):
    for root, _, files in os.walk(DATA):
        for f in files:
            if f.upper().startswith(name) and f.upper().endswith(".XPT"):
                df = pd.read_sas(os.path.join(root, f))
                num = df.select_dtypes("number").columns
                df[num] = df[num].mask(df[num].abs() < 1e-10, 0)
                print("Loaded", f, df.shape)
                return df
    print("NOT FOUND:", name)

demo = load("DEMO_L")
cbc = load("CBC_L")
ghb = load("GHB_L")
diq = load("DIQ_L")

for n, d in [("DEMO", demo), ("CBC", cbc), ("GHB", ghb)]:
    print(n, [c for c in d.columns if c.startswith(("WT", "SDMV"))])

Loaded DEMO_L.xpt (11933, 27)
Loaded CBC_L.xpt (8727, 23)
Loaded GHB_L.xpt (7199, 3)
Loaded DIQ_L.xpt (11744, 9)
DEMO ['WTINT2YR', 'WTMEC2YR', 'SDMVSTRA', 'SDMVPSU']
CBC ['WTPH2YR']
GHB ['WTPH2YR']


In [3]:
df = (demo
      .merge(cbc, on="SEQN")
      .merge(ghb.drop(columns="WTPH2YR"), on="SEQN")
      .merge(diq[["SEQN", "DIQ010"]], on="SEQN", how="left"))

coh = df[(df["RIDAGEYR"] >= 18)
         & (df["WTPH2YR"] > 0)
         & df["LBXGH"].notna()
         & ~df["DIQ010"].isin([1, 7, 9])
         & (df["RIDEXPRG"] != 1)].copy()

print("Merged:", len(df), "| Cohort:", len(coh))

Merged: 7199 | Cohort: 5144


In [4]:
rows = []
for cut in [5.7, 6.5]:
    y = (coh["LBXGH"] >= cut).astype(int)
    rows.append({
        "Outcome": f"HbA1c >= {cut}%",
        "Cases (n)": int(y.sum()),
        "Unweighted %": round(100 * y.mean(), 1),
        "Weighted %": round(100 * np.average(y, weights=coh["WTPH2YR"]), 1),
    })
pd.DataFrame(rows)

,Outcome,Cases (n),Unweighted %,Weighted %
0,HbA1c >= 5.7%,1552,30.2,26.4
1,HbA1c >= 6.5%,152,3.0,2.5
